# 12webdata: data files for the two globe pages

Part D of PF5. Turns the results of Parts A and B into small files that the web pages in `docs/` load. No downloads.

| Section | Files in `docs/data/` | What |
|---|---|---|
| 2 | `event/smoke_NNN.png`, `season/smoke_NNN.png` | one transparent picture of smoke PM2.5 per time step (event 3-hourly at 0.25 deg, season daily at 0.5 deg) |
| 3 | `event/wind.json`, `season/wind.json` | short wind trails (where the air moves over the next hours), for the flowing-lines effect |
| 4 | `event/fires.json`, `season/fires.json` | fire hotspots per day |
| 5 | `event/meta.json`, `season/meta.json` | times, map corners, legend, storm tracks and the numbers shown in the side panel |

Together about 10 to 20 MB, fine for GitHub Pages. Re-run whenever a setting below changes.

## Section 0: settings

In [1]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
EVENT_RAW = PROJECT_DIR / "data" / "raw" / "event"
EVENT_PROC = PROJECT_DIR / "data" / "processed" / "event"
SEASON_RAW = PROJECT_DIR / "data" / "raw" / "season"
SEASON_PROC = PROJECT_DIR / "data" / "processed" / "season"
WEB_DIR = PROJECT_DIR / "docs" / "data"

SMOKE_LEVELS = [1, 2, 5, 10, 15, 25, 35, 55.5, 100, 250]  # <- VISUAL: colour steps (micrograms per m3); below the first step is transparent
SMOKE_ALPHA = [60, 90, 120, 150, 175, 195, 210, 225, 235, 245]  # <- VISUAL: opacity of each step (0 to 255)
SMOKE_CMAP = "YlOrBr"  # <- VISUAL: colours from light yellow to dark brown (reads well over the dark Blue Marble map)

WIND_SEED_DEG = 3.0  # <- VISUAL: spacing of the event wind trail starting points, degrees
SEASON_SEED_DEG = 4.0  # <- VISUAL: spacing for the season trails (wider: there are 86 days of them)
EVENT_WIND_LEVEL = 850  # <- ERA5 pressure level for the event trails (hPa)
EVENT_WIND_EVERY_H = 6  # <- a new set of trails every n hours
EVENT_TRAIL_H = 18  # <- each trail follows the air for n hours
SEASON_TRAIL_H = 24  # <- season trails (10 m daily wind), hours
FIRE_CELL_EVENT = 0.1  # <- event hotspots are counted in cells of this size (degrees)

STORM_NAMES = None  # <- None = all storms of the season; or a list such as ["CHOI-WAN", "CHAMPI"]

## Section 1: imports and data

In [2]:
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib
from PIL import Image

warnings.filterwarnings("ignore", message="Ambiguous reference date string")
for sub in ["event", "season"]:
    (WEB_DIR / sub).mkdir(parents=True, exist_ok=True)
R = 6371000.0

ev = xr.merge([xr.open_dataset(EVENT_RAW / f"geosfp_{v}.nc") for v in ["ocsmass", "brsmass", "bcsmass"]])
ev_smoke = (ev["ocsmass"] + ev["brsmass"] + ev["bcsmass"]) * 1e9
EV_T = pd.to_datetime(ev_smoke.time.values)
se = xr.open_dataset(SEASON_PROC / "geosfp_daily.nc")
se_smoke = se["smoke"].load()
SE_T = pd.to_datetime(se_smoke.time.values)
print("Event:", len(EV_T), "steps", EV_T[0], "to", EV_T[-1], " grid", ev_smoke.shape[1:])
print("Season:", len(SE_T), "days", SE_T[0].date(), "to", SE_T[-1].date(), " grid", se_smoke.shape[1:])

Event: 80 steps 2026-09-28 01:30:00 to 2026-10-07 22:30:00  grid (241, 225)
Season: 86 days 2026-08-01 to 2026-10-25  grid (121, 113)


## Section 2: smoke pictures

Each picture is the smoke field on the model grid, north up, coloured with the steps in Section 0 and transparent where the air is clean. The pages stretch it over the map between the corners stored in `meta.json`.

In [3]:
cmap = matplotlib.colormaps[SMOKE_CMAP]
LUT = np.zeros((len(SMOKE_LEVELS) + 1, 4), dtype=np.uint8)
legend = []
for k in range(len(SMOKE_LEVELS)):
    rgb = (np.array(cmap(0.15 + 0.85 * k / (len(SMOKE_LEVELS) - 1))[:3]) * 255).astype(np.uint8)
    LUT[k + 1, :3] = rgb
    LUT[k + 1, 3] = SMOKE_ALPHA[k]
    legend.append({"from": SMOKE_LEVELS[k], "rgba": [int(c) for c in rgb] + [SMOKE_ALPHA[k]]})


def to_png(field, path):
    a = np.nan_to_num(np.asarray(field, dtype="float64"), nan=0.0)
    idx = np.searchsorted(SMOKE_LEVELS, a, side="right")
    img = LUT[idx][::-1]
    Image.fromarray(img, "RGBA").save(path, optimize=True)


def corners(da):
    lat = da.lat.values
    lon = da.lon.values
    dlat = (lat[1] - lat[0]) / 2
    dlon = (lon[1] - lon[0]) / 2
    return {"west": float(lon[0] - dlon), "east": float(lon[-1] + dlon), "south": float(lat[0] - dlat), "north": float(lat[-1] + dlat)}


for i in range(len(EV_T)):
    to_png(ev_smoke.isel(time=i).values, WEB_DIR / "event" / f"smoke_{i:03d}.png")
for i in range(len(SE_T)):
    to_png(se_smoke.isel(time=i).values, WEB_DIR / "season" / f"smoke_{i:03d}.png")
size_ev = sum(p.stat().st_size for p in (WEB_DIR / "event").glob("smoke_*.png")) / 1e6
size_se = sum(p.stat().st_size for p in (WEB_DIR / "season").glob("smoke_*.png")) / 1e6
print(f"Event pictures: {len(EV_T)} ({size_ev:.1f} MB); season pictures: {len(SE_T)} ({size_se:.1f} MB)")

Event pictures: 80 (1.2 MB); season pictures: 86 (0.4 MB)


## Section 3: wind trails

Short paths of air parcels, started on a regular grid and moved with the ERA5 wind (the same method as `03trajectories`). The pages draw them as moving, fading lines, a little like earth.nullschool.net. Event: 850 hPa wind (about 1.5 km up, where most smoke travels), a new set every 6 hours. Season: 10 m daily wind, a new set every day.

In [4]:
def make_trails(u, v, t_hours, lat, lon, starts_h, trail_h, seed_deg, step_min=30, keep_every_min=90):
    # Heun integration; returns a list of [t0_hours, [lon, lat, lon, lat, ...]]
    LA = np.arange(lat.min() + seed_deg / 2, lat.max(), seed_deg)
    LO = np.arange(lon.min() + seed_deg / 2, lon.max(), seed_deg)
    s_lat, s_lon = [g.ravel() for g in np.meshgrid(LA, LO, indexing="ij")]

    def wind(h, y, x):
        k = np.clip(np.searchsorted(t_hours, h) - 1, 0, len(t_hours) - 2)
        ft = np.clip((h - t_hours[k]) / (t_hours[k + 1] - t_hours[k]), 0, 1)
        j = np.clip(np.searchsorted(lat, y) - 1, 0, len(lat) - 2)
        i = np.clip(np.searchsorted(lon, x) - 1, 0, len(lon) - 2)
        fy = np.clip((y - lat[j]) / (lat[j + 1] - lat[j]), 0, 1)
        fx = np.clip((x - lon[i]) / (lon[i + 1] - lon[i]), 0, 1)
        out = []
        for f in (u, v):
            vals = []
            for kk in (k, k + 1):
                a = f[kk, j, i] * (1 - fx) + f[kk, j, i + 1] * fx
                b = f[kk, j + 1, i] * (1 - fx) + f[kk, j + 1, i + 1] * fx
                vals.append(a * (1 - fy) + b * fy)
            out.append(vals[0] * (1 - ft) + vals[1] * ft)
        return out

    def move(y, x, uu, vv, dt):
        return y + np.rad2deg(vv * dt / R), x + np.rad2deg(uu * dt / (R * np.cos(np.deg2rad(y))))

    trails = []
    n_steps = int(trail_h * 60 / step_min)
    keep = keep_every_min // step_min
    for h0 in starts_h:
        y = s_lat.copy()
        x = s_lon.copy()
        h = np.full(y.shape, float(h0))
        pts = [np.stack([x, y], axis=1)]
        for n in range(n_steps):
            u1, v1 = wind(h, y, x)
            yp, xp = move(y, x, u1, v1, step_min * 60)
            u2, v2 = wind(h + step_min / 60, yp, xp)
            y, x = move(y, x, (u1 + u2) / 2, (v1 + v2) / 2, step_min * 60)
            h = h + step_min / 60
            if (n + 1) % keep == 0:
                pts.append(np.stack([x, y], axis=1))
        arr = np.stack(pts, axis=1)
        inside = (arr[:, :, 0] >= lon.min()) & (arr[:, :, 0] <= lon.max()) & (arr[:, :, 1] >= lat.min()) & (arr[:, :, 1] <= lat.max())
        for p in range(arr.shape[0]):
            ok = inside[p]
            m = int(np.argmin(ok)) if not ok.all() else len(ok)
            if m >= 3:
                trails.append([round(float(h0), 2), np.round(arr[p, :m].ravel(), 2).tolist()])
    return trails


era = xr.concat([xr.open_dataset(f) for f in sorted(EVENT_RAW.glob("era5_levels_*.nc"))], dim="valid_time").sortby("latitude")
eu = era["u"].sel(pressure_level=EVENT_WIND_LEVEL).values
evv = era["v"].sel(pressure_level=EVENT_WIND_LEVEL).values
e_hours = (pd.to_datetime(era.valid_time.values) - EV_T[0]).total_seconds().values / 3600
ev_starts = np.arange(0, (EV_T[-1] - EV_T[0]).total_seconds() / 3600 - EVENT_TRAIL_H / 2, EVENT_WIND_EVERY_H)
ev_trails = make_trails(eu, evv, e_hours, era.latitude.values, era.longitude.values, ev_starts, EVENT_TRAIL_H, WIND_SEED_DEG)
(WEB_DIR / "event" / "wind.json").write_text(json.dumps({"step_hours": 1.5, "level": f"{EVENT_WIND_LEVEL} hPa", "trails": ev_trails}, separators=(",", ":")))

e5 = xr.open_dataset(SEASON_PROC / "era5_daily.nc")
s_hours = (pd.to_datetime(e5.time.values) + pd.Timedelta(hours=12) - SE_T[0]).total_seconds().values / 3600
se_starts = np.arange(0, len(SE_T) * 24 - SEASON_TRAIL_H / 2, 24)
se_trails = make_trails(e5["u10"].values, e5["v10"].values, s_hours, e5.lat.values, e5.lon.values, se_starts, SEASON_TRAIL_H, SEASON_SEED_DEG, step_min=60, keep_every_min=120)
(WEB_DIR / "season" / "wind.json").write_text(json.dumps({"step_hours": 2.0, "level": "10 m (daily mean)", "trails": se_trails}, separators=(",", ":")))
print(f"Event trails: {len(ev_trails)} ({(WEB_DIR / 'event' / 'wind.json').stat().st_size / 1e6:.1f} MB); season trails: {len(se_trails)} ({(WEB_DIR / 'season' / 'wind.json').stat().st_size / 1e6:.1f} MB)")

Event trails: 17440 (2.9 MB); season trails: 21930 (3.7 MB)


## Section 4: fire hotspots

Event: VIIRS hotspots (nominal and high confidence, all three VIIRS satellites) counted per 0.1 degree cell and day. Season: NOAA-20 VIIRS hotspots per 0.5 degree cell and day (from `05season`). Each entry is [longitude, latitude, number of hotspots].

In [5]:
f = pd.read_csv(EVENT_RAW / "firms_event.csv", low_memory=False)
f = f[f["source"].str.startswith("VIIRS") & (f["confidence"].astype(str) != "l")]
f["date"] = pd.to_datetime(f["acq_date"])
f["cx"] = (np.floor(f["longitude"] / FIRE_CELL_EVENT) + 0.5) * FIRE_CELL_EVENT
f["cy"] = (np.floor(f["latitude"] / FIRE_CELL_EVENT) + 0.5) * FIRE_CELL_EVENT
ev_days = pd.date_range(EV_T[0].normalize(), EV_T[-1].normalize())
ev_fires = {}
for d in ev_days:
    g = f[f["date"] == d].groupby(["cx", "cy"]).size()
    ev_fires[f"{d:%Y-%m-%d}"] = [[round(x, 2), round(y, 2), int(n)] for (x, y), n in g.items()]
(WEB_DIR / "event" / "fires.json").write_text(json.dumps(ev_fires, separators=(",", ":")))

fc = pd.read_csv(SEASON_RAW / "firms_cells_VIIRS_NOAA20_NRT.csv", parse_dates=["date"])
se_fires = {}
for d in SE_T:
    g = fc[fc["date"] == d]
    se_fires[f"{d:%Y-%m-%d}"] = [[round(x, 2), round(y, 2), int(n)] for x, y, n in zip(g["cell_lon"], g["cell_lat"], g["hotspots"])]
(WEB_DIR / "season" / "fires.json").write_text(json.dumps(se_fires, separators=(",", ":")))
print(f"Event fire cells per day (mean): {np.mean([len(v) for v in ev_fires.values()]):.0f} ({(WEB_DIR / 'event' / 'fires.json').stat().st_size / 1e6:.1f} MB)")
print(f"Season fire cells per day (mean): {np.mean([len(v) for v in se_fires.values()]):.0f} ({(WEB_DIR / 'season' / 'fires.json').stat().st_size / 1e6:.1f} MB)")

Event fire cells per day (mean): 5022 (0.9 MB)
Season fire cells per day (mean): 944 (1.4 MB)


## Section 5: times, storms and panel numbers (`meta.json`)

In [6]:
tracks = pd.read_csv(SEASON_PROC / "season_tracks.csv", parse_dates=["time"])
tracks["wind"] = pd.to_numeric(tracks["USA_WIND"], errors="coerce")
if STORM_NAMES:
    tracks = tracks[tracks["NAME"].isin(STORM_NAMES)]


def storm_list(t0, t1):
    out = []
    for sid, g in tracks.groupby("SID"):
        g = g[(g["time"] >= t0 - pd.Timedelta(days=1)) & (g["time"] <= t1 + pd.Timedelta(days=1))].sort_values("time")
        if len(g) < 2:
            continue
        out.append({"name": g["NAME"].iloc[0].title(), "max_wind_kt": float(g["wind"].max()), "t_hours": [round((t - t0).total_seconds() / 3600, 2) for t in g["time"]], "lon": [round(float(x), 2) for x in g["LON"]], "lat": [round(float(y), 2) for y in g["LAT"]], "wind_kt": [None if np.isnan(w) else float(w) for w in g["wind"]]})
    return out


em = pd.read_csv(EVENT_PROC / "event_metrics.csv", parse_dates=["time"])
ev_meta = {"title": "Choi-wan event", "start": EV_T[0].isoformat(), "step_hours": 3, "n": len(EV_T), "times": [t.strftime("%Y-%m-%d %H:%M") for t in EV_T], "bounds": corners(ev_smoke), "legend": legend, "storms": storm_list(EV_T[0], EV_T[-1])}
ev_meta["panel"] = {"corridor_smoke_area_km2": [round(float(x)) for x in em["corridor_smoke_area_km2"]], "storm_wind_kt": [None if np.isnan(x) else round(float(x)) for x in em["storm_wind_kt"]], "near_storm_seasalt": [None if np.isnan(x) else round(float(x), 1) for x in em["near_storm_seasalt"]]}
(WEB_DIR / "event" / "meta.json").write_text(json.dumps(ev_meta, separators=(",", ":")))

sd = pd.read_csv(SEASON_PROC / "season_daily_series.csv", header=[0, 1], index_col=0, parse_dates=True)
se_meta = {"title": "2026 haze season", "start": SE_T[0].isoformat(), "step_hours": 24, "n": len(SE_T), "times": [t.strftime("%Y-%m-%d") for t in SE_T], "bounds": corners(se_smoke), "legend": legend, "storms": storm_list(SE_T[0], SE_T[-1])}
se_meta["panel"] = {k: [round(float(x), 1) for x in sd[(place, "smoke")].reindex(SE_T).values] for k, place in [("kl_smoke", "Kuala Lumpur"), ("sg_smoke", "Singapore"), ("sumatra_smoke", "Sumatra"), ("kalimantan_smoke", "Kalimantan"), ("corridor_smoke", "Pacific corridor")]}
se_meta["panel"]["typhoon_day"] = [bool(x) for x in sd[("All", "typhoon_day")].reindex(SE_T).values]
(WEB_DIR / "season" / "meta.json").write_text(json.dumps(se_meta, separators=(",", ":")))
print("Event storms:", [s["name"] for s in ev_meta["storms"]])
print("Season storms:", len(se_meta["storms"]))
total = sum(p.stat().st_size for p in WEB_DIR.rglob("*") if p.is_file()) / 1e6
print(f"docs/data total: {total:.1f} MB")

Event storms: ['Nolo', 'Surigae', 'Choi-Wan', 'Koguma']
Season storms: 22
docs/data total: 10.6 MB


___
**END**